## Data Loading

In [2]:
#CARREGAR O DATASET
from datasets import load_dataset

#damos como parametro o nome do repositorio do dataset(que esta no hugging face)
dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/266k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/930 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [3]:
#METADADOS QUE DEFINEM COMO ESTA ORGANIZADO O DATASET
dataset_raw["train"].features


{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

## Data Pre-Processing

In [4]:
#CRIAMOS O TOKENIZER ESPECIFICO DE PT (divide em tokense converte para numeros)
from transformers import AutoTokenizer

#passamos como parametro o nome do modelo que vamos usar
#queremos usar o tokenizer deste modelo
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
inputs = tokenizer("As aulas de PLNEB são muito interessantes!")
inputs
#o tokenizer esta a partir a string em tokens, e passa os para ids
#os ids tem haver com a frequencia ( quanto maior a frequencia menor sera o id da palavra)

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [6]:
tokenizer.convert_ids_to_tokens(inputs['input_ids'])
# como PLNEB é uma palavra muito rara ( o modelo nao conhece) ele vai tentar partir em subwords que conhece
# por exemplo, caso escrevamos mal uma palavra ele vai fazer o mesmo
# o SEP serve para separar as frases

['[CLS]',
 'As',
 'aulas',
 'de',
 'P',
 '##L',
 '##N',
 '##EB',
 'são',
 'muito',
 'interessantes',
 '!',
 '[SEP]']

In [7]:
#o dataset é uma lista de tokens por isso vamos passar lhe uma lista de tokens
#ou seja podemos passar uma string ou uma lista de tokens
tokens =["as", "aulas", "plneb","são", "interessantes", "!"]
inputs=tokenizer(tokens, is_split_into_words=True)

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [8]:
len(tokens), len(new_tokens)

(6, 10)

In [9]:
inputs.word_ids()
# da nos o indice dos tokens na frase

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [24]:
#quando palavras são divididas em subwords coloca na primeira subword a label real
#nas restantes subwords coloca -100
def align_labels_with_tokens(word_ids,labels):
    new_labels= []
    previous_word = None
    for word_id in word_ids:
        #se for None significa que sao os caracteres especiais(no inicio e no fim)
        # que nao pertencem a frase (foram adicionados pelo modelo)
        if word_id == None:
            new_labels.append(-100)
        elif previous_word != word_id:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        previous_word = word_id
    return new_labels


def tokenize_dataset(dataset):
    res = []
    for row in dataset:
        inputs = tokenizer(
            row["tokens"],
            is_split_into_words=True,
            truncation=True,        # Cortar se for muito grande
            max_length=512          # O limite máximo do BERT
        )
        new_labels = align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels
        res.append(inputs)
    return res

train_data = tokenize_dataset(dataset_raw["train"])
test_data = tokenize_dataset(dataset_raw["test"])
len(train_data), len(test_data)


(3716, 930)

In [25]:
#transforma as frases todas com o mesmo tamanho (mais eficiente para o GPU)
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Avaliar

In [14]:
#!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a921d2df8301e4eaea756c225a856552b6d7ba26c46c0c6e65a417a01cfd01e3
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [26]:
import evaluate

seqeval = evaluate.load("seqeval")

In [27]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Model Training

In [28]:
# Isto vai aos metadados do dataset, procura a coluna "ner_tags" e extrai os "names"
label_list = dataset_raw["train"].features["ner_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}


In [29]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [31]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="meu_modelo_ner_pt",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.067213,0.949736,0.968978,0.959260,0.985244
2,No log,0.070435,0.947368,0.968661,0.957896,0.984675
3,0.026800,0.072100,0.945645,0.969294,0.957324,0.984806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=699, training_loss=0.02421924416428813, metrics={'train_runtime': 308.3995, 'train_samples_per_second': 36.148, 'train_steps_per_second': 2.267, 'total_flos': 658641856754904.0, 'train_loss': 0.02421924416428813, 'epoch': 3.0})

# Avaliar


In [32]:
text="""Donald Trump terminou a sua intervenção na Casa Branca dizendo que os EUA vão conseguir obter o urânio enriquecido do Irão.
"Nós vamos conseguir", garantiu. O presidente dos EUA disse ainda ser "improvável" que os seus enviados especiais Steve Witkoff e
Jared Kushner se desloquem para negociar com os iranianos."Há uma boa hipótese de este problema terminar.
Se não terminar, teremos de voltar a bombardeá-los, sem dó nem piedade", reafirmou.
"""

In [34]:
from transformers import pipeline

classifier = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple" )
classifier(text)

[{'entity_group': 'Pessoa',
  'score': np.float32(0.68671143),
  'word': 'Donald Trump',
  'start': 0,
  'end': 12},
 {'entity_group': 'Organizacao',
  'score': np.float32(0.66352236),
  'word': 'Casa',
  'start': 43,
  'end': 47},
 {'entity_group': 'Local',
  'score': np.float32(0.7054689),
  'word': 'EUA',
  'start': 70,
  'end': 73},
 {'entity_group': 'Local',
  'score': np.float32(0.9105797),
  'word': 'EUA',
  'start': 175,
  'end': 178},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.99349993),
  'word': 'Steve Witkoff',
  'start': 239,
  'end': 252},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.99276155),
  'word': 'Jar',
  'start': 256,
  'end': 259},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.9713723),
  'word': '##ed Kushner',
  'start': 259,
  'end': 269}]